# 基于 BERT 的 IMDB 影评情感分类实战

## 1. 内容概述

BERT（Bidirectional Encoder Representations from Transformers）是 Google 于 2018 年末发布的预训练语言模型。与传统的单向语言模型不同，BERT 基于 Transformer 的 Encoder 并采用双向结构，能够同时利用上下文的左侧和右侧信息进行表征学习，在文本分类、问答、命名实体识别等自然语言处理任务中表现优异。

BERT 的核心创新点集中在预训练方法上，使用了 **Masked Language Model（MLM）** 和 **Next Sentence Prediction（NSP）** 两种自监督任务，分别捕捉词语级别和句子级别的表征。预训练完成后，开发者可在这些权重之上添加任务特定的分类头，通过 Fine-tuning 快速适配下游任务。

本案例演示如何在昇腾 NPU 上，基于预训练的 **bert-base-uncased** 模型，在 **IMDB 影评数据集**上完成情感二分类任务（正面 / 负面），内容涵盖：

- CANN 环境加载与 NPU 可用性验证
- BERT 模型介绍与加载
- IMDB 数据集介绍、下载与 Tokenize 预处理
- 训练超参数配置与模型训练
- 测试集准确率评估
- 训练前后推理效果对比，直观展现 Fine-tuning 的价值

## 2. 整体流程概览

本案例将完整走通在昇腾 NPU 上基于预训练 BERT 进行文本情感分类的全流程：

```
环境准备 → 数据集下载与预处理 → 模型加载（NPU） → 训练 → 测试集评估 → 训练前后推理效果对比
```

各环节说明：

| 环节 | 说明 |
|------|------|
| 环境准备 | 加载 CANN 环境变量，安装 Python 依赖，验证 NPU 可用性 |
| 数据集下载与预处理 | 下载 IMDB 数据集，将原始文本 Tokenize 为 input_ids / attention_mask |
| 模型加载 | 加载 bert-base-uncased 预训练权重与随机初始化的分类头到 NPU |
| 训练 | 配置 TrainingArguments 超参数，用 Trainer 在 IMDB 训练集上 Fine-tuning |
| 测试集评估 | 在测试集上计算模型准确率 |
| 训练前后推理效果对比 | 对比未微调模型与微调后模型在同一组示例文本上的推理结果 |

下面逐步实现每个环节。

## 3. 环境准备与依赖安装

本案例依赖以下核心组件：

| 库名 | 用途 |
|------|------|
| `torch` + `torch_npu` | PyTorch 深度学习框架 + 昇腾 NPU 适配 |
| `transformers` | Hugging Face 模型加载与训练（BERT、Trainer 等） |
| `accelerate` | 训练加速与设备调度 |
| `tqdm` | 进度条可视化 |
| `numpy` | 数值计算与指标统计 |

### 3.1 加载 CANN 环境

> ⚠️ **必须在 `import torch` 之前执行此单元格！**
> Jupyter 内核不继承 shell 环境变量，缺这步会导致 `torch_npu` 加载失败。

In [ ]:
import os, subprocess, warnings
warnings.filterwarnings('ignore')

_cann_paths = [
    os.path.join(os.environ.get("ASCEND_TOOLKIT_HOME", ""), "set_env.sh"),
    "/home/developer/Ascend/cann-9.2.0/set_env.sh",
    "/usr/local/Ascend/ascend-toolkit/set_env.sh",
]
_cann = next((p for p in _cann_paths if p and os.path.exists(p)), None)
if _cann is None:
    raise FileNotFoundError("找不到 CANN set_env.sh，请确认 CANN 安装路径")

print(f"正在加载 CANN 环境: {_cann}")
result = subprocess.run(["bash", "-c", f"source {_cann} && env"], capture_output=True, text=True, timeout=10)
for line in result.stdout.splitlines():
    if "=" in line:
        k, v = line.split("=", 1)
        os.environ[k] = v

import sys
sys.modules['torchaudio'] = None

print(f"CANN 环境已加载 (ASCEND_TOOLKIT_HOME={os.environ.get('ASCEND_TOOLKIT_HOME','?')})")

### 3.2 安装 Python 依赖

若环境中已预装所需库，可跳过此步骤。

In [ ]:
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "torch>=2.5", "torch_npu", "transformers>=4.40", 
     "accelerate>=0.30", "tqdm>=4.60", "numpy>=1.24"],
    check=True,
)

### 3.3 验证 NPU 环境

运行以下代码，确认昇腾 NPU 可用。如果输出 NPU 设备信息，说明环境配置正确。

> 💡 **小贴士**：`torch_npu` 是昇腾 NPU 的 PyTorch 适配层。导入它之后，PyTorch 就能自动识别并使用 NPU 进行计算，使用方式与 `torch.cuda` 非常相似。例如：
> - `torch.npu.set_device(0)` 等价于 `torch.cuda.set_device(0)`
> - `model.to("npu")` 等价于 `model.to("cuda")`

In [ ]:
import os

os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

import torch
from torch.utils.data import Dataset, DataLoader

import sys, types, importlib.util
if "torchaudio" not in sys.modules:
    _fake_ta = types.ModuleType("torchaudio")
    _fake_ta.__spec__ = importlib.util.spec_from_loader("torchaudio", loader=None)
    _fake_ta.__version__ = "0.0.0"
    sys.modules["torchaudio"] = _fake_ta

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
    EvalPrediction,
)
import numpy as np
from tqdm import tqdm

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    try:
        import torch_npu  # noqa: F401
        if torch.npu.is_available():
            device = torch.device("npu")
        else:
            device = torch.device("cpu")
    except ImportError:
        device = torch.device("cpu")

print(f"当前设备: {device}")
if str(device) == "npu":
    print(f"NPU 设备数量: {torch.npu.device_count()}")
    print(f"当前 NPU 设备: {torch.npu.current_device()}")
    print(f"NPU 设备名称: {torch.npu.get_device_name(0)}")

## 4. 模型介绍

### 4.1 BERT 模型简介

BERT 全称是来自变换器的双向编码器表征量（Bidirectional Encoder Representations from Transformers），其主要创新点在预训练方法上，即用了 **Masked Language Model** 和 **Next Sentence Prediction** 两种方法分别捕捉词语和句子级别的表征。

**① Masked Language Model（MLM）**

在用 MLM 方法训练 BERT 的时候，随机把语料库中 15% 的单词做 Mask 操作。对于这 15% 的单词做 Mask 操作分为三种情况：

- 80% 的单词直接用 `[Mask]` 替换
- 10% 的单词直接替换成另一个新的单词
- 10% 的单词保持不变

这样模型无法确定哪个位置被遮蔽，被迫学习整个句子的上下文表征。

**② Next Sentence Prediction（NSP）**

因为涉及到 Question Answering（QA）和 Natural Language Inference（NLI）之类的任务，增加了 NSP 预训练任务，目的是让模型理解两个句子之间的联系。训练的输入是句子 A 和 B，B 有一半的几率是 A 的下一句，输入这两个句子，BERT 模型预测 B 是不是 A 的下一句。

**③ 预训练权重与 Fine-tuning**

BERT 预训练之后，会保存它的 Embedding table 和 12 层 Transformer 权重（BERT-BASE）或 24 层 Transformer 权重（BERT-LARGE）。使用预训练好的 BERT 模型可以对下游任务进行 Fine-tuning，比如：文本分类、相似度判断、阅读理解等。

本案例使用 **bert-base-uncased** 作为基座模型，在其基础上添加一个随机初始化的线性分类头，通过在 IMDB 数据集上 Fine-tuning 完成情感二分类任务。

### 4.2 加载预训练 BERT 模型

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
model.to(device)

print(f"模型已加载到 {device}")
print(f"模型参数量: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

> 💡 **关键代码解读**：
> - `BertTokenizer`：将文本转化为 BERT 能理解的 token 序列（input_ids、attention_mask 等）
> - `BertForSequenceClassification`：在 BERT 编码器之上添加线性分类头，`num_labels=2` 对应正面 / 负面两类
> - `model.to(device)`：将模型加载到 NPU 上进行计算
>
> 加载时会看到 `MISSING` 提示（`classifier.weight` / `classifier.bias`），这是正常现象——预训练权重中不含分类头参数，分类头会被随机初始化，这正是后续 Fine-tuning 需要训练的部分。

## 5. 数据集介绍与下载

### 5.1 IMDB 数据集简介

IMDB 情感分析数据集是 NLP 领域广泛使用的电影评论情感分类任务数据集，最初由斯坦福大学发布，用于评估文本分类模型的情感识别能力。

- **数据规模**：训练集 25000 条，测试集 25000 条（各 50% 正面，50% 负面），另有 50000 条未标注评论可用于无监督学习
- **数据格式**：每条数据由 text（评论文本）和 label（1=正面，0=负面）组成
- **目录结构**：`aclImdb/train/pos/`、`aclImdb/train/neg/`、`aclImdb/test/pos/`、`aclImdb/test/neg/`，每个子目录下为 `.txt` 格式的评论文件

### 5.2 下载与解压数据集

In [ ]:
import os
import subprocess

os.makedirs("data", exist_ok=True)
subprocess.run(
    ["wget", "-P", "data", "-N", "--no-check-certificate",
     "https://mindspore-website.obs.myhuaweicloud.com/notebook/datasets/aclImdb_v1.tar.gz"],
    check=True,
)
subprocess.run(["tar", "-xzf", "data/aclImdb_v1.tar.gz", "-C", "data/"], check=True)
print("数据集下载并解压完成")

### 5.3 数据预处理

大语言模型无法直接处理原始文本，需要将文本通过 tokenizer 转化为数字 token 序列。数据预处理分为两步：

1. **`format_imdb_dataset`**：遍历 `aclImdb` 目录下的 `.txt` 文件，返回 `(label_str, text)` 对
2. **`IMDBTorchDataset`**：将文本通过 `BertTokenizer` 编码为 `input_ids` 和 `attention_mask`，并转换为 PyTorch 张量

**关键参数说明：**

| 参数 | 含义 |
|------|------|
| `padding="max_length"` | 将所有序列填充至固定长度，便于批量处理 |
| `truncation=True` | 超过最大长度的文本将被截断 |
| `max_length=128` | 最大序列长度设为 128，平衡显存占用与信息保留 |

In [ ]:
def format_imdb_dataset(split):
    """Read aclImdb directory and yield (label_str, text) pairs."""
    base_dir = os.path.join("./data/aclImdb/", split)
    for label_str in ("pos", "neg"):
        label_dir = os.path.join(base_dir, label_str)
        if not os.path.isdir(label_dir):
            continue
        for fname in sorted(os.listdir(label_dir)):
            if not fname.endswith(".txt"):
                continue
            with open(os.path.join(label_dir, fname), "r", encoding="utf-8") as f:
                text = f.read()
            yield label_str, text


class IMDBTorchDataset(Dataset):
    def __init__(self, split, tokenizer, max_seq_len=128):
        self.examples = []
        for label_str, text in format_imdb_dataset(split):
            label = 1 if label_str == "pos" else 0
            encoded = tokenizer(
                text, padding="max_length", truncation=True, max_length=max_seq_len
            )
            self.examples.append(
                {
                    "input_ids": encoded["input_ids"],
                    "attention_mask": encoded["attention_mask"],
                    "labels": label,
                }
            )

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        item = self.examples[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(item["labels"], dtype=torch.long),
        }

In [ ]:
dataset_train = IMDBTorchDataset("train", tokenizer)
dataset_val = IMDBTorchDataset("test", tokenizer)
dataset_test = IMDBTorchDataset("test", tokenizer)

print(f"训练集大小: {len(dataset_train)}")
print(f"验证集大小: {len(dataset_val)}")
print(f"测试集大小: {len(dataset_test)}")

## 6. 训练模型与代码介绍

### 6.1 代码结构

训练部分代码分为以下模块：

| 模块 | 作用 |
|------|------|
| `compute_metrics` | 评估指标函数，计算模型在验证集上的准确率（Accuracy） |
| `TrainingArguments` | 训练超参数配置，包括 batch_size、学习率、训练轮数等 |
| `Trainer` | Hugging Face 提供的训练器，封装训练循环、日志记录与 checkpoint 保存 |
| `evaluate_fn` | 测试集评估函数，在测试集上批量推理并计算准确率 |

### 6.2 配置评估指标与训练超参数

In [ ]:
def compute_metrics(eval_pred: EvalPrediction):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": float((predictions == labels).mean())}

training_args = TrainingArguments(
    output_dir="./output_imdb",
    per_device_train_batch_size=64,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    num_train_epochs=1,
    logging_steps=200,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
)

print("训练参数配置:")
print(f"  Batch Size: {training_args.per_device_train_batch_size}")
print(f"  学习率: {training_args.learning_rate}")
print(f"  训练轮数: {training_args.num_train_epochs}")
print(f"  日志步数: {training_args.logging_steps}")
print(f"  评估策略: {training_args.eval_strategy}")
print(f"  保存策略: {training_args.save_strategy}")

**超参数解读：**

| 参数 | 值 | 含义 |
|------|-----|------|
| `per_device_train_batch_size` | 64 | 每张 NPU 上的训练 batch size |
| `per_device_eval_batch_size` | 32 | 每张 NPU 上的评估 batch size |
| `learning_rate` | 2e-5 | 学习率，BERT 微调常用的小学习率 |
| `num_train_epochs` | 1 | 训练轮数，整个训练集遍历 1 次 |
| `logging_steps` | 200 | 每 200 步打印一次训练日志 |
| `eval_strategy` | epoch | 每个 epoch 结束时在验证集上评估 |
| `save_strategy` | epoch | 每个 epoch 结束时保存 checkpoint |

### 6.3 启动训练

In [ ]:
trainer = Trainer(
    model=model,
    train_dataset=dataset_train,
    eval_dataset=dataset_val,
    compute_metrics=compute_metrics,
    args=training_args,
)

trainer.train()

训练过程中你会看到类似如下的输出：

```
{'loss': 0.4025, 'learning_rate': 2e-05, 'epoch': 0.5}
{'loss': 0.2829, 'learning_rate': 2e-05, 'epoch': 1.0}
```

- **loss（损失值）**：衡量模型预测与正确答案的差距，应随训练逐渐下降
- **learning_rate（学习率）**：控制每次参数更新的步长
- **epoch（轮次）**：当前训练进度，1.0 表示已遍历一遍全部训练数据

### 6.4 测试集评估

训练完成后，在测试集上评估模型整体准确率。

In [ ]:
def evaluate_fn(model, test_dataset, batch_size=32):
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    epoch_acc = 0
    step_total = 0
    model.eval()
    with tqdm(total=len(test_loader)) as progress_bar:
        for batch in test_loader:
            labels = batch.pop("labels")
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = labels.to(device)
            with torch.no_grad():
                logits = model(**batch).logits
            acc = compute_metrics((logits.cpu().numpy(), labels.cpu().numpy()))["accuracy"]
            epoch_acc += acc
            step_total += 1
            progress_bar.update(1)
    return epoch_acc / step_total

acc = evaluate_fn(model, dataset_test)
print(f"Test Accuracy: {acc:.4f}")

## 7. 训练前后推理效果对比

下面通过同一组示例文本，对比 **训练前**（随机初始化分类头）和 **训练后** 模型的推理效果，直观展现 Fine-tuning 的价值。

### 7.1 定义推理函数

In [ ]:
label_map = {0: "negative", 1: "positive"}

def predict(text, model, tokenizer):
    encoded = tokenizer(text, padding="max_length", truncation=True, max_length=128)
    input_ids = torch.tensor([encoded["input_ids"]], dtype=torch.long, device=device)
    attention_mask = torch.tensor([encoded["attention_mask"]], dtype=torch.long, device=device)
    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    predicted_label = np.argmax(logits.cpu().numpy(), axis=-1)[0]
    return label_map[predicted_label]

test_texts = [
    "Terrible script and awful acting.",
    "Boring, predictable and a complete waste of time.",
]
expected = ["negative", "negative"]

### 7.2 训练前推理（重新加载未微调的原始模型）

重新加载一份未经微调的 `bert-base-uncased` 模型（分类头随机初始化），观察其在示例文本上的表现。

In [ ]:
untrained_model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
untrained_model.to(device)

print("===== 训练前推理结果 =====")
for text, exp in zip(test_texts, expected):
    pred = predict(text, untrained_model, tokenizer)
    print(f"Text: '{text}'")
    print(f"  期望: {exp}  |  预测: {pred}")
    print()

### 7.3 训练后推理

使用经过 Fine-tuning 的模型对同一组示例文本进行推理。

In [ ]:
print("===== 训练后推理结果 =====")
for text, exp in zip(test_texts, expected):
    pred = predict(text, model, tokenizer)
    print(f"Text: '{text}'")
    print(f"  期望: {exp}  |  预测: {pred}")
    print()

### 7.4 对比总结

| 文本 | 期望 | 训练前预测 | 训练后预测 |
|------|------|-----------|-----------|
| Terrible script and awful acting. | negative | positive（错误） | negative |
| Boring, predictable and a complete waste of time. | negative | positive（错误） | negative |

**结论：**

训练前由于分类头是随机初始化的，模型对文本情感的预测结果不可靠，几乎等同于随机猜测；经过在 IMDB 数据集上的 Fine-tuning 后，模型能够准确识别影评的情感倾向（正面 / 负面），验证了 **预训练 + 微调** 范式的有效性。

这也体现了 BERT 的核心价值：通过大规模预训练学习通用语言表征，再通过少量下游任务数据 Fine-tuning 即可快速适配具体任务，显著降低标注数据需求与训练成本。